# Infer hybrid — Qwen3.5-2B + adapter phiên 3 (KHÔNG SFT lại)

Chỉ đổi retrieval so với lượt 3 (BM25 + rerank, METEOR 0,5539): thêm dense e5-small + RRF.
Cùng adapter `tot_nhat` phiên 3, cùng `--max-new 1024 --gen-bs 2` → chênh điểm là do retrieval.
Lệch phân phối: adapter train trên prompt BM25+rerank, nay nhận prompt hybrid.


In [ ]:
import os, sys, shutil, zipfile, subprocess, json, gc
from pathlib import Path

SRC = None
for cand in Path("/kaggle/input").glob("**/*"):
    if (cand.name == "legalqa" and (cand / "bm25.py").exists()
            and "legalqa-v10-src" in str(cand)):
        SRC = cand.parent.parent
        break
assert SRC, "Không thấy codebase — gắn dataset legalqa-v10-src"
REPO = Path("/kaggle/working/repo")
shutil.rmtree(REPO, ignore_errors=True)
shutil.copytree(SRC, REPO)
os.chdir(REPO)
sys.path.insert(0, "src"); sys.path.insert(0, "scripts")
print("codebase:", (REPO / "COMMIT.txt").read_text().strip())

!pip -q install transformers==5.14.1 peft nltk
!pip -q uninstall -y torchao

# Adapter phiên 3 được train trên prompt CÓ số hiệu và KHÔNG trần ký tự. Cả hai phải
# khớp, không phải một trong hai: sai bất kỳ cái nào là đưa model vào phân phối nó chưa
# từng thấy, và kết quả tệ đi mà không có dấu vết nào trong log.
os.environ["LEGALQA_SO_HIEU"] = "1"
from legalqa.prompt import CO_SO_HIEU
from build_sft import TOP_N, MAX_CHARS
assert CO_SO_HIEU is True, "prompt đang TẮT số hiệu — sai checkpoint phiên 3"
assert TOP_N == 5 and MAX_CHARS > 10**6, f"codebase CŨ: top_n={TOP_N} max_chars={MAX_CHARS}"
assert (REPO / "scripts/infer.py").exists()
print(f"số hiệu ON | top_n={TOP_N} max_chars={MAX_CHARS:,} (không trần)")

In [ ]:
ch = next((p for p in Path("/kaggle/input").glob("**/chunks.jsonl")), None)
assert ch, "Không thấy chunks.jsonl — gắn dataset legalqa-clean"
CLEAN = Path("/kaggle/working/clean")
CLEAN.mkdir(parents=True, exist_ok=True)
for f in ch.parent.iterdir():
    d = CLEAN / f.name
    if not d.exists():
        d.symlink_to(f)
os.environ["CLEAN_DIR"] = str(CLEAN)
for n in ("chunks.jsonl", "bm25.pkl", "corpus.jsonl"):
    assert (CLEAN / n).exists(), f"thiếu {n} trong legalqa-clean"

# Tập test BTC: dict {id: {question, answer=None}}, KHÔNG phải jsonl của ta.
# `nap_cau_hoi()` nhận cả hai dạng theo đuôi file nên đường dẫn phải giữ đuôi .json.
CAU_HOI = next(Path("/kaggle/input").glob("**/public-official.json"), None)
assert CAU_HOI, "Không thấy public-official.json — gắn dataset official"
_d = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert len(_d) == 1000, f"tập test có {len(_d)} câu, phải là 1.000"
assert all(v.get("answer") is None for v in _d.values()), "tập test có đáp án?"
print(CAU_HOI, len(_d), "câu")

BASE = next((p.parent for p in Path("/kaggle/input").glob("**/tokenizer.json")
             if "qwen" in str(p).lower() and (p.parent / "config.json").exists()), None)
assert BASE, "Không thấy model nền Qwen — gắn model qwen3-5-2b-base"
idx = BASE / "model.safetensors.index.json"
can = sorted(set(json.loads(idx.read_text())["weight_map"].values())) if idx.exists() else []
if [n for n in can if not (BASE / n).exists()]:
    FIX = Path("/kaggle/working/base")
    FIX.mkdir(exist_ok=True)
    for f in BASE.iterdir():
        if not (FIX / f.name).exists():
            (FIX / f.name).symlink_to(f)
    thuc = sorted(BASE.glob("*.safetensors"))
    assert len(thuc) == len(can), f"{len(thuc)} shard thật vs {len(can)} index cần"
    for ten, that in zip(can, thuc):
        if not (FIX / ten).exists():
            (FIX / ten).symlink_to(that)
    BASE = FIX
    print("vá tên shard ->", BASE)

# Adapter phiên 3 (train trên prompt BM25+rerank — lượt 3 nộp 0,5539). Lọc theo tên dataset: gắn nhiều phiên thì glob bắt trúng
# phien2 trước và ta suy luận bằng checkpoint cũ mà không hay.
ADAPTER = next((q.parent for q in Path("/kaggle/input").glob("**/tot_nhat/adapter_config.json")
                if "phien3" in str(q)), None)
assert ADAPTER, "không thấy tot_nhat/ của phiên 3 — gắn dataset legalqa-adapter-phien3"

local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "XLMRobertaForSequenceClassification" in p.read_text()]
if local:
    RERANK = str(local[0])
else:
    from huggingface_hub import snapshot_download
    RERANK = snapshot_download("AITeamVN/Vietnamese_Reranker")
print("base   ", BASE)
print("adapter", ADAPTER)
print("rerank ", RERANK)

In [ ]:
# Dense: e5-small 118M. §15.8 chốt max_len 256 — `nap_dense()` trong infer.py ép sẵn,
# không truyền từ đây. Tải từ HF vì 118M rẻ hơn một vòng publish dataset.
local = [p.parent for p in Path("/kaggle/input").glob("**/config.json")
         if "e5-small" in str(p).lower() or "multilingual-e5" in str(p).lower()]
if local:
    DENSE = str(local[0])
else:
    from huggingface_hub import snapshot_download
    DENSE = snapshot_download("intfloat/multilingual-e5-small")
print("dense  ", DENSE)

# Resume CÓ ĐIỀU KIỆN. Prompt của lượt hybrid khác prompt lượt BM25 thuần, nên nối file
# của lượt 4 vào là trộn hai phân phối retrieval trong cùng một tập đo — hybrid có lãi
# hay không sẽ không đọc được nữa. Lọc theo "hybrid" trong đường dẫn dataset, và vì
# `legalqa-clean` cũng có thể chứa file tên giống nên lọc cả hai điều kiện.
RA = Path("/kaggle/working/out"); RA.mkdir(parents=True, exist_ok=True)
for ten in ("prompts.jsonl", "du_doan.jsonl"):
    cu = next((q for q in Path("/kaggle/input").glob(f"**/{ten}") if "hybrid" in str(q)), None)
    if cu:
        shutil.copy(cu, RA / ten)
        print(f"nối {ten}: {sum(1 for _ in open(RA / ten, encoding='utf-8')):,} dòng")
    else:
        print(f"không có {ten} của lượt hybrid trước — pha tương ứng chạy từ đầu")

# Ngân sách. Pha 1: encode dense 436s + retrieval ~2s/câu (BM25 deep 1.000 + dense matmul
# 0,1s + RRF + rerank 1,58s) = ~40 phút cho 1.000 câu; `--gio-pha1 1.2` là đệm.
# Pha 2: `--max-new 1024` — TRẦN CỦA LƯỢT 3, không phải 1536. Lượt 4 nâng lên 1536 và
# điểm TỤT 0,5539 -> 0,5524: 109 câu chạm trần 1.024 không thiếu đuôi mà đang LẶP (89,0%
# có một 12-gram lặp >=3 lần ngay ở trần 1.024; đuôi thêm là "khớp khớp khớp..."). Recall
# đã bão hoà nên thêm token chỉ kéo precision, và alpha=0,9 vẫn để 10% trọng số cho nó.
# Trần 1.024 đang làm phanh cho vòng lặp; gỡ phanh phải đi kèm adapter hết lặp.
# gen-bs 2 như lượt 3 (29,3 s/câu đo được) => 1.000 câu ~8,1 giờ + pha 1 ~0,7 giờ.
# gen-bs 4 chưa từng chạy với adapter phiên 3 — OOM giữa pha 2 là mất cả phiên.
# `--gio-pha2 9.8` là trần để kịp commit output trước mốc 12 giờ; thiếu thì publish
# `legalqa-du-doan-hybrid` rồi chạy lượt 2 — hai pha đều ghi tăng dần.
subprocess.run([sys.executable, "scripts/infer.py",
    "--cau-hoi", str(CAU_HOI),
    "--model", str(BASE), "--adapter", str(ADAPTER), "--rerank", RERANK,
    "--dense", DENSE,
    "--ra", "/kaggle/working/out", "--submission",
    "--batch-size", "32", "--pool", "50",
    "--max-new", "1024", "--gen-bs", "2",
    "--gio-pha1", "1.2", "--gio-pha2", "9.8",
], check=True)


In [ ]:
# Không có gold nên không chấm được. Nghiệm thu đúng thứ quyết định điểm nộp: HÌNH
# DẠNG submission, rồi mới đến chẩn đoán bịa (so với b318 để biết fix số hiệu có ăn).
import re
from collections import Counter

sub = json.loads(Path("/kaggle/working/out/submission.json").read_text(encoding="utf-8"))
goc = json.loads(Path(CAU_HOI).read_text(encoding="utf-8"))
assert set(sub) == set(goc), f"khoá lệch: {len(set(goc) - set(sub))} thiếu, {len(set(sub) - set(goc))} lạ"
assert all(isinstance(v, dict) and isinstance(v.get("answer"), str) for v in sub.values())
rong = [k for k, v in sub.items() if not v["answer"].strip()]
print(f"submission: {len(sub)} khoá | rỗng {len(rong)}")
assert not rong, f"{len(rong)} câu chưa sinh được — tăng --gio-pha2 rồi chạy lại kernel"

rows = [json.loads(l) for l in open("/kaggle/working/out/du_doan.jsonl", encoding="utf-8")]
pr = {json.loads(l)["id"]: json.loads(l)["prompt"]
      for l in open("/kaggle/working/out/prompts.jsonl", encoding="utf-8")}
NAM = re.compile(r"\b(?:19|20)\d{2}\b")
DIEU = re.compile(r"Điều\s+(\d+[a-zA-Z]?)")
st = Counter()
for x in rows:
    p, c = str(x["du_doan"]), pr[x["id"]]
    pn, cn = set(NAM.findall(p)), set(NAM.findall(c))
    pd, cd = set(DIEU.findall(p)), set(DIEU.findall(c))
    st["n"] += 1
    # Không có gold nên KHÔNG đo được `nam_bia_SAI`. Chỉ còn `nam_bia` = năm trong dự
    # đoán mà prompt không có. Ghi rõ để đừng ai so con số này với 46,7% của dev.
    st["nam_bia"] += bool(pn - cn)
    st["dieu_bia"] += bool(pd - cd)
    st["cut"] += not p.rstrip().endswith((".", "!", "?", '"', ":", ";"))
    st["prompt_co_nam"] += bool(cn)
n = st["n"]
print(f"\ntrên {n} câu test:")
for k in ("nam_bia", "dieu_bia", "cut", "prompt_co_nam"):
    print(f"  {k:14} {st[k]:4}  {st[k]/n:6.1%}")
print("\nb318 (CHƯA có số hiệu) trên 500 câu bỏ qua: nam_bia 62,6%  dieu_bia 10,4%  cut 10,8%")
print("b318 trên 120 câu dev:                        nam_bia 65,0%  dieu_bia  3,3%  cut  6,7%")
print("prompt_co_nam là mẫu số thật: dev cũ chỉ 56,7% prompt có năm nào.")